In [1]:
!pip install langchain_huggingface


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install tf-keras


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableParallel
import os

In [4]:
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [6]:
llm = ChatGroq(
    api_key=groq_api_key,  
    model="llama-3.3-70b-versatile",
    temperature=.7
)

In [8]:
docs = [
    "LangChain is a framework to build applications powered by LLMs.",
    "LangChain supports agents, tools, and RAG workflows.",
    "LangChain enables document-based Q&A systems with FAISS and OpenAI.",
    "To use RAG, you must first index your documents using embeddings.",
    "Document chunking helps improve retrieval accuracy by dividing text into smaller units."
]


In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(docs, embeddings)
retriever = vectorstore.as_retriever()

In [15]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

qa_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
) | RunnablePassthrough.assign(
    answer=(
        {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
        | prompt | llm | StrOutputParser()
    )
)



**search_type="similarity" and k=4**

k = how many chunks get retrieved and stuffed into the LLM's prompt as context.

In [11]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

**MMR instead**

fetch_k: how many candidates to pull initially before the diversity filter runs (e.g., pull top 20, then MMR-select the best/most diverse 4 from those 20)
lambda_mult: 0 to 1, controls the relevance/diversity tradeoff. 1.0 = pure relevance (behaves like plain similarity search). 0.0 = pure diversity (maximize spread, ignore relevance). 0.5 is a balanced default.

In [14]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5}
)

In [ ]:
# Output with MMR
response = qa_chain.invoke("What is LangChain used for?")
print("Answer:", response["answer"])
print("\nSources:")
for doc in response["context"]:
    print("-", doc.page_content)

Answer: LangChain is used to build applications powered by LLMs (Large Language Models). More specifically, it enables document-based Q&A systems.

Sources:
- LangChain is a framework to build applications powered by LLMs.
- Document chunking helps improve retrieval accuracy by dividing text into smaller units.
- LangChain enables document-based Q&A systems with FAISS and OpenAI.


Above We can see That MMR based retrieval got differen chuck retrieved for same prompt

In [ ]:
# Output with similarity
response = qa_chain.invoke("What is LangChain used for?")
print("Answer:", response["answer"])
print("\nSources:")
for doc in response["context"]:
    print("-", doc.page_content)

Answer: LangChain is used to build applications powered by LLMs (Large Language Models). It supports various features such as agents, tools, and RAG workflows, and enables document-based Q&A systems.

Sources:
- LangChain is a framework to build applications powered by LLMs.
- LangChain supports agents, tools, and RAG workflows.
- LangChain enables document-based Q&A systems with FAISS and OpenAI.


Above We can see That similarity based retrieval got similar chuck retrieved

In [ ]:
# Simple
response = qa_chain.invoke("What is LangChain used for?")
print(response)
